# Multimodal Stress Detection Pipeline

In [9]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
subprocess.run(['pip','install','-q','pyarrow','scikit-learn','librosa',
    'soundfile','opencv-python-headless','transformers','accelerate','bitsandbytes','seaborn'])
print('Done.')

Mounted at /content/drive
Done.


In [10]:
import sys, os
FRAMEWORK_DIR = '/content/drive/MyDrive/stress_framework'
sys.path.insert(0, FRAMEWORK_DIR)
os.chdir(FRAMEWORK_DIR)
print('CWD:', os.getcwd())

CWD: /content/drive/MyDrive/stress_framework


In [ ]:
import tensorflow as tf, keras
print('TF:', tf.__version__, '| Keras:', keras.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

TF: 2.19.0 | Keras: 3.13.2
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from preprocessing import load_parquet
df = load_parquet()
print('Shape:', df.shape)
print(df[['label_anger','label_fear','label_happy','label_sad','label_surprise','label_disgust']].describe())

Shape: (23248, 12)
        label_anger    label_fear   label_happy     label_sad  label_surprise  \
count  23248.000000  23248.000000  23248.000000  23248.000000    23248.000000   
mean       0.155727      0.041000      0.480070      0.157318        0.047832   
std        0.362184      0.152557      0.629320      0.324379        0.168053   
min        0.000000      0.000000      0.000000      0.000000        0.000000   
25%        0.000000      0.000000      0.000000      0.000000        0.000000   
50%        0.000000      0.000000      0.333333      0.000000        0.000000   
75%        0.000000      0.000000      0.666667      0.333333        0.000000   
max        3.000000      1.666667      3.000000      3.000000        3.000000   

       label_disgust  
count   23248.000000  
mean        0.119552  
std         0.319725  
min         0.000000  
25%         0.000000  
50%         0.000000  
75%         0.000000  
max         3.000000  


In [ ]:
from preprocessing import build_datasets
train_ds, val_ds, test_ds, scaler, df_test, arrays = build_datasets(df)
t_te, a_te, v_te, y_te = arrays
print('Train:', len(train_ds), '| Val:', len(val_ds), '| Test:', len(test_ds))

Train: 509 | Val: 73 | Test: 146


In [ ]:
from models import build_model
from config import MODEL_NAMES, FUSION_STRATEGIES
for name in MODEL_NAMES:
    for fusion in FUSION_STRATEGIES:
        m = build_model(name, fusion)
        print(f'{m.name:<35} params={m.count_params():,}')

[MulT/early] params=282,438
MulT_early                          params=282,438
[MulT/late] params=199,494
MulT_late                           params=199,494
[MulT/cross_modal] params=646,214
MulT_cross_modal                    params=646,214
[TFN/early] params=1,448,054
TFN_early                           params=1,448,054
[TFN/late] params=199,494
TFN_late                            params=199,494
[TFN/cross_modal] params=1,448,054
TFN_cross_modal                     params=1,448,054
[LMF/early] params=203,722
LMF_early                           params=203,722
[LMF/late] params=199,494
LMF_late                            params=199,494
[LMF/cross_modal] params=203,722
LMF_cross_modal                     params=203,722
[MFN/early] params=282,438
MFN_early                           params=282,438
[MFN/late] params=644,294
MFN_late                            params=644,294
[MFN/cross_modal] params=644,294
MFN_cross_modal                     params=644,294
[SelfMM/early] params=309,144
Sel

In [ ]:
from train import train_one
result = train_one('LMF', 'cross_modal', train_ds, val_ds)
print(result)

  LMF / cross_modal
[LMF/cross_modal] params=203,722
Epoch 1/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.9717 - mae: 0.7984
Epoch 1: val_loss improved from None to 0.18721, saving model to /content/drive/MyDrive/checkpoints/LMF_cross_modal_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/checkpoints/LMF_cross_modal_best.keras
509/509 ━━━━━━━━━━━━━━━━━━━━ 19s 20ms/step - loss: 0.4895 - mae: 0.4881 - val_loss: 0.1872 - val_mae: 0.2442 - learning_rate: 0.0010
Epoch 2/50
502/509 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1826 - mae: 0.2333
Epoch 2: val_loss improved from 0.18721 to 0.16489, saving model to /content/drive/MyDrive/checkpoints/LMF_cross_modal_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/checkpoints/LMF_cross_modal_best.keras
509/509 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.1813 - mae: 0.2286 - val_loss: 0.1649 - val_mae: 0.2099 - learning_rate: 0.0010
Epoch 3/50
502/509 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1679 - 

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
    function keepAlive() {
        document.querySelector("#top-toolbar > colab-connect-button")
                ?.shadowRoot?.querySelector("div")?.click();
    }
    setInterval(keepAlive, 60000);
'''))
from train import train_all
results = train_all(train_ds, val_ds)
if results:
    import pandas as pd
    print(pd.DataFrame(results)[['model_name','fusion','best_val_loss','elapsed_sec']])

<IPython.core.display.Javascript object>

⏭️  Skipping MulT_early
⏭️  Skipping MulT_late
⏭️  Skipping MulT_cross_modal
⏭️  Skipping TFN_early
⏭️  Skipping TFN_late
⏭️  Skipping TFN_cross_modal
⏭️  Skipping LMF_early
⏭️  Skipping LMF_late
⏭️  Skipping LMF_cross_modal
⏭️  Skipping MFN_early
⏭️  Skipping MFN_late
⏭️  Skipping MFN_cross_modal
⏭️  Skipping SelfMM_early
⏭️  Skipping SelfMM_late
⏭️  Skipping SelfMM_cross_modal


In [ ]:
# ══ STEP 1 — Delete all 15 bad checkpoints ══
import os, glob
from config import CHECKPOINT_DIR

ckpts = glob.glob(os.path.join(CHECKPOINT_DIR, '*_best.keras'))
for path in ckpts:
    os.remove(path)
    print(f"  Deleted: {os.path.basename(path)}")
print(f"\n Cleared {len(ckpts)} old checkpoints. Ready to retrain.")

  Deleted: MulT_early_best.keras
  Deleted: LMF_cross_modal_best.keras
  Deleted: MulT_late_best.keras
  Deleted: MulT_cross_modal_best.keras
  Deleted: TFN_early_best.keras
  Deleted: TFN_late_best.keras
  Deleted: TFN_cross_modal_best.keras
  Deleted: LMF_early_best.keras
  Deleted: LMF_late_best.keras
  Deleted: MFN_early_best.keras
  Deleted: MFN_late_best.keras
  Deleted: MFN_cross_modal_best.keras
  Deleted: SelfMM_early_best.keras
  Deleted: SelfMM_late_best.keras
  Deleted: SelfMM_cross_modal_best.keras

 Cleared 15 old checkpoints. Ready to retrain.


In [ ]:
# ══ STEP 2 — Retrain all 15 from scratch (keep-alive included) ══
from IPython.display import display, Javascript
display(Javascript('''
    function keepAlive() {
        document.querySelector("#top-toolbar > colab-connect-button")
                ?.shadowRoot?.querySelector("div")?.click();
    }
    setInterval(keepAlive, 60000);
'''))

from train import train_all
results = train_all(train_ds, val_ds)

import pandas as pd
if results:
    print(pd.DataFrame(results)[['model_name','fusion','best_val_loss','elapsed_sec']])

<IPython.core.display.Javascript object>

⏭️  Skipping MulT_early
⏭️  Skipping MulT_late
⏭️  Skipping MulT_cross_modal
⏭️  Skipping TFN_early
⏭️  Skipping TFN_late
⏭️  Skipping TFN_cross_modal
⏭️  Skipping LMF_early
⏭️  Skipping LMF_late
⏭️  Skipping LMF_cross_modal
⏭️  Skipping MFN_early
⏭️  Skipping MFN_late
⏭️  Skipping MFN_cross_modal
⏭️  Skipping SelfMM_early
⏭️  Skipping SelfMM_late
⏭️  Skipping SelfMM_cross_modal


In [ ]:
import keras
keras.config.enable_unsafe_deserialization()

# Now reload evaluate and run
from evaluate import full_comparison
df_results = full_comparison(test_ds)
print(df_results)

/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis 3 of a tensor of shape (32, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis 3 of a tensor of shape (11, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(



 Model      Fusion    MSE    MAE   RMSE     F1
SelfMM        late 0.0885 0.1627 0.2776 0.9283
SelfMM cross_modal 0.0891 0.1618 0.2786 0.9287
  MulT       early 0.0890 0.1642 0.2787 0.9303
   MFN        late 0.0894 0.1657 0.2788 0.9287
   LMF        late 0.0898 0.1673 0.2792 0.9287
   MFN cross_modal 0.0897 0.1652 0.2793 0.9293
SelfMM       early 0.0897 0.1627 0.2794 0.9279
   TFN        late 0.0896 0.1646 0.2795 0.9289
   MFN       early 0.0898 0.1630 0.2797 0.9286
  MulT        late 0.0902 0.1667 0.2802 0.9292
  MulT cross_modal 0.0914 0.1668 0.2816 0.9288
   TFN       early 0.0915 0.1636 0.2822 0.9303
   TFN cross_modal 0.0920 0.1639 0.2824 0.9287
   LMF cross_modal 0.0922 0.1695 0.2828 0.9269
   LMF       early 0.0922 0.1690 0.2830 0.9284
     Model       Fusion     MSE     MAE    RMSE      F1
0   SelfMM         late  0.0885  0.1627  0.2776  0.9283
1   SelfMM  cross_modal  0.0891  0.1618  0.2786  0.9287
2     MulT        early  0.0890  0.1642  0.2787  0.9303
3      MFN         late

In [ ]:
from stress_score import stress_calc
import numpy as np
print(f'Formula: S = A*(1-V)/2  range:[0,1]  levels: Low/Moderate/Severe')
for intensities, label in [
    ([3.0,3.0,0.0,0.0,0.0,0.0],'Anger+Fear'),
    ([0.0,0.0,3.0,0.0,0.0,0.0],'Happy'),
    ([1.5,1.0,0.5,0.5,0.2,0.3],'Mixed'),
    ([0.1,0.1,0.1,0.1,0.1,0.1],'All low')]:
    r = stress_calc.compute(np.array(intensities))
    print(f'{label:<15} V={r["valence"]:>6.3f} A={r["arousal"]:>6.3f} S={r["stress_score"]:>6.3f}  {r["stress_label"]}')

Formula: S = A*(1-V)/2  range:[0,1]  levels: Low/Moderate/Severe
Anger+Fear      V=-0.650 A= 0.750 S= 0.619  Moderate
Happy           V= 0.800 A= 0.400 S= 0.040  Low
Mixed           V=-0.412 A= 0.550 S= 0.388  Moderate
All low         V=-0.233 A= 0.433 S= 0.267  Low


In [ ]:
from main import run_rl
rl_result = run_rl(df_test, arrays)
print('A2C final:        ', round(sum(rl_result['history']['A2C'][-300:])/300, 3))
print('Dueling DQN final:', round(sum(rl_result['history']['DuelingDQN'][-300:])/300, 3))

## Inference Cells — Adaptive Multi-Turn Conversation

In [ ]:
from main import run_inference
result = run_inference(
    text='I feel very anxious and overwhelmed at work.',
    model_name='LMF', fusion='cross_modal', run_chat=True)

In [ ]:
# ══ CELL 12B — Audio Inference ══
from google.colab import files
from main import run_inference
print('Upload audio (.wav/.mp3/.m4a):')
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]
result = run_inference(audio_path=audio_path, model_name='LMF', fusion='cross_modal', run_chat=True)

In [ ]:
# ══ CELL 12C — Video Inference (auto splits facial+audio) ══
import subprocess
subprocess.run(['apt-get','install','-qq','-y','ffmpeg'])
from google.colab import files
from main import run_inference
print('Upload video (.mp4/.mov/.avi):')
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
result = run_inference(video_path=video_path, model_name='LMF', fusion='cross_modal', run_chat=True)
if result.get('audio_note'): print(f'\n{result["audio_note"]}')

In [ ]:
# ══ CELL 12D — All 3 Modalities ══
from main import run_inference
result = run_inference(
    text='I feel anxious and cannot sleep.',
    audio_path=audio_path, video_path=video_path,
    model_name='LMF', fusion='cross_modal', run_chat=True)

In [ ]:
# ══ CELL 13 — Generate All Charts ══
import pandas as pd, numpy as np, os
from train import load_model
from evaluate import predict
from stress_score import stress_calc
from visualize import generate_all_charts
from config import RESULTS_DIR
df_results = pd.read_csv(os.path.join(RESULTS_DIR,'full_comparison.csv'))
best = df_results.iloc[0]
best_name, best_fusion = best['Model'], best['Fusion']
model = load_model(best_name, best_fusion)
y_true, y_pred = predict(model, test_ds, best_name=='SelfMM')
stress_results = stress_calc.compute_from_predictions(y_pred)
generate_all_charts(df_results=df_results, y_true=y_true, y_pred=y_pred,
    stress_results=stress_results, best_model=best_name, best_fusion=best_fusion)

## CHFN Novel Model + External SOTA Comparison

In [ ]:
# ══ CELL N1 — Build CHFN ══
from novel_model import build_chfn, compile_chfn, CHFN_CUSTOM_OBJECTS
from models import build_model
chfn = build_chfn(n_bag=3, use_contrastive=True)
chfn = compile_chfn(chfn, learning_rate=1e-3)
print('='*55)
print('  CHFN — Cross-modal Hierarchical Fusion Network')
print('='*55)
print(f'  Total parameters: {chfn.count_params():,}')
lmf = build_model('LMF','cross_modal')
print(f'  LMF/cross_modal : {lmf.count_params():,} params (best baseline)')

  CHFN — Cross-modal Hierarchical Fusion Network
  Total parameters: 1,778,356
[LMF/cross_modal] params=203,722
  LMF/cross_modal : 203,722 params (best baseline)


In [ ]:
import os
from config import CHECKPOINT_DIR
path = os.path.join(CHECKPOINT_DIR, "CHFN_best.keras")
if os.path.exists(path):
    os.remove(path)
    print("  Deleted CHFN checkpoint — ready to retrain")

  Deleted CHFN checkpoint — ready to retrain


In [ ]:
# ══ CELL N2 — Train CHFN (fixed callbacks) ══
import os, time, keras
from novel_model import build_chfn, compile_chfn, wrap_dataset_chfn, CHFN_CUSTOM_OBJECTS
from config import CHECKPOINT_DIR, RESULTS_DIR, EPOCHS, PATIENCE, LR_PATIENCE, LR_FACTOR

CHFN_CKPT = os.path.join(CHECKPOINT_DIR, "CHFN_best.keras")

if os.path.exists(CHFN_CKPT):
    print("Loading existing CHFN checkpoint...")
    chfn = keras.models.load_model(CHFN_CKPT, custom_objects=CHFN_CUSTOM_OBJECTS)
else:
    chfn = build_chfn(n_bag=3, use_contrastive=True)
    chfn = compile_chfn(chfn, learning_rate=1e-3)
    print(f"Parameters: {chfn.count_params():,}")

    tr_chfn = wrap_dataset_chfn(train_ds)
    va_chfn = wrap_dataset_chfn(val_ds)

    callbacks = [
        keras.callbacks.ModelCheckpoint(
            CHFN_CKPT,
            monitor="val_main_loss",
            mode="min",               # ← fix
            save_best_only=True,
            verbose=1),
        keras.callbacks.EarlyStopping(
            monitor="val_main_loss",
            mode="min",               # ← fix
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_main_loss",
            mode="min",               # ← fix
            factor=LR_FACTOR,
            patience=LR_PATIENCE,
            min_lr=1e-6,
            verbose=1),
        keras.callbacks.CSVLogger(
            os.path.join(RESULTS_DIR, "CHFN_training_log.csv")),
    ]

    t0      = time.time()
    history = chfn.fit(tr_chfn, validation_data=va_chfn,
                       epochs=EPOCHS, callbacks=callbacks, verbose=1)
    elapsed = (time.time() - t0) / 60
    best    = min(history.history["val_main_loss"])
    print(f"\nDone in {elapsed:.1f} min | best val_main_loss: {best:.4f}")

Parameters: 1,778,356


Cause: Unable to locate the source code of <function wrap_dataset_chfn.<locals>._expand at 0x7b34e5de1940>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: lineno is out of bounds
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function wrap_dataset_chfn.<locals>._expand at 0x7b34e5de1940>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: lineno is out of bounds
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function wrap_dataset_chfn.<locals>._expand at 0x7b34e5de19e0>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: lineno is out of bounds
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function wrap_dataset_chfn.<locals>._expand at 0x7b34e5de19e0>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: lineno is out of bounds
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis 3 of a tensor of shape (None, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


509/509 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - aux_a_loss: 0.2945 - aux_t_loss: 0.3950 - aux_v_loss: 0.3233 - head_0_loss: 0.7886 - head_1_loss: 0.8005 - head_2_loss: 0.7389 - loss: 1.3679 - main_loss: 0.6138
Epoch 1: val_main_loss improved from None to 0.12800, saving model to /content/drive/MyDrive/checkpoints/CHFN_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/checkpoints/CHFN_best.keras
509/509 ━━━━━━━━━━━━━━━━━━━━ 87s 74ms/step - aux_a_loss: 0.1823 - aux_t_loss: 0.2050 - aux_v_loss: 0.1912 - head_0_loss: 0.3480 - head_1_loss: 0.3508 - head_2_loss: 0.3350 - loss: 0.6973 - main_loss: 0.2686 - val_aux_a_loss: 0.1289 - val_aux_t_loss: 0.1279 - val_aux_v_loss: 0.1230 - val_head_0_loss: 0.1318 - val_head_1_loss: 0.1299 - val_head_2_loss: 0.1273 - val_loss: 0.3929 - val_main_loss: 0.1280 - learning_rate: 0.0010
Epoch 2/50
508/509 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - aux_a_loss: 0.1354 - aux_t_loss: 0.1333 - aux_v_loss: 0.1309 - head_0_loss: 0.1453 - head_1_loss: 0.1432 

In [ ]:
# ══ CELL N3 — Evaluate CHFN ══
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, f1_score
from config import EMOTION_NAMES, EMOTION_PRESENCE_THRESHOLD, RESULTS_DIR
import os, pandas as pd
all_true, all_pred = [], []
for inputs, labels in test_ds:
    out = chfn(inputs, training=False)
    all_pred.append(out['main'].numpy()); all_true.append(labels.numpy())
y_true_chfn = np.vstack(all_true); y_pred_chfn = np.vstack(all_pred)
print('\nCHFN Per-Emotion Results:')
print(f'{"Emotion":<12} {"RMSE":>8} {"MAE":>8} {"F1":>8}')
print('-'*40)
chfn_metrics = {}
for i, name in enumerate(EMOTION_NAMES):
    yt,yp=y_true_chfn[:,i],y_pred_chfn[:,i]
    rmse=float(np.sqrt(mean_squared_error(yt,yp))); mae=float(mean_absolute_error(yt,yp))
    f1=float(f1_score((yt>=EMOTION_PRESENCE_THRESHOLD).astype(int),(yp>=EMOTION_PRESENCE_THRESHOLD).astype(int),average='weighted',zero_division=0))
    chfn_metrics[name]={'RMSE':rmse,'MAE':mae,'F1':f1}
    print(f'{name:<12} {rmse:>8.4f} {mae:>8.4f} {f1:>8.4f}')
avg_rmse=np.mean([chfn_metrics[n]['RMSE'] for n in EMOTION_NAMES])
avg_mae =np.mean([chfn_metrics[n]['MAE']  for n in EMOTION_NAMES])
avg_f1  =np.mean([chfn_metrics[n]['F1']   for n in EMOTION_NAMES])
print('-'*40); print(f'{"AVERAGE":<12} {avg_rmse:>8.4f} {avg_mae:>8.4f} {avg_f1:>8.4f}')
chfn_row={'Model':'CHFN (Ours)','Fusion':'hierarchical',
          'MSE':round(avg_rmse**2,4),'MAE':round(avg_mae,4),'RMSE':round(avg_rmse,4),'F1':round(avg_f1,4)}
print('\nCHFN row:', chfn_row)

/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis 3 of a tensor of shape (32, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis 3 of a tensor of shape (11, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(



CHFN Per-Emotion Results:
Emotion          RMSE      MAE       F1
----------------------------------------
anger          0.3101   0.1828   0.9140
fear           0.1490   0.0521   0.9881
happy          0.4843   0.3402   0.8225
sad            0.3002   0.1923   0.9280
surprise       0.1669   0.0657   0.9878
disgust        0.2823   0.1396   0.9318
----------------------------------------
AVERAGE        0.2821   0.1621   0.9287

CHFN row: {'Model': 'CHFN (Ours)', 'Fusion': 'hierarchical', 'MSE': np.float64(0.0796), 'MAE': np.float64(0.1621), 'RMSE': np.float64(0.2821), 'F1': np.float64(0.9287)}


In [ ]:
# Delete old SOTA checkpoints (built with Lambda layers)
import os
from config import CHECKPOINT_DIR

for name in ["MISA_best.keras", "CTNet_best.keras", "UniMSE_best.keras"]:
    path = os.path.join(CHECKPOINT_DIR, name)
    if os.path.exists(path):
        os.remove(path)
        print(f" Deleted: {name}")
print("Done  re-run the SOTA training cell.")

 Deleted: MISA_best.keras
 Deleted: CTNet_best.keras
 Deleted: UniMSE_best.keras
Done  re-run the SOTA training cell.


In [ ]:
import keras
keras.config.enable_unsafe_deserialization()

In [ ]:
# CELL S1–S3 — Train MISA / CTNet / UniMSE
import os, time, json, keras
from sota_models import (build_misa, compile_misa, wrap_misa,
                         build_ctnet, compile_ctnet,
                         build_unimse, compile_unimse, wrap_unimse,
                         SOTA_CUSTOM_OBJECTS)
from config import CHECKPOINT_DIR, RESULTS_DIR, EPOCHS, PATIENCE, LR_PATIENCE, LR_FACTOR

def train_external(model_obj, ckpt_path, tr_ds, va_ds, monitor="val_main_loss"):
    if os.path.exists(ckpt_path):
        print(f"  Checkpoint found: {os.path.basename(ckpt_path)}")
        return keras.models.load_model(ckpt_path, custom_objects=SOTA_CUSTOM_OBJECTS)

    callbacks = [
        keras.callbacks.ModelCheckpoint(
            ckpt_path,
            monitor=monitor,
            mode="min",           # ← fix
            save_best_only=True,
            verbose=0),
        keras.callbacks.EarlyStopping(
            monitor=monitor,
            mode="min",           # ← fix
            patience=PATIENCE,
            restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(
            monitor=monitor,
            mode="min",           # ← fix
            factor=LR_FACTOR,
            patience=LR_PATIENCE,
            min_lr=1e-6),
    ]

    t0  = time.time()
    his = model_obj.fit(tr_ds, validation_data=va_ds,
                        epochs=EPOCHS, callbacks=callbacks, verbose=1)
    best = min(his.history.get(monitor, [99]))
    print(f"  Done {(time.time()-t0)/60:.1f} min | best {monitor}: {best:.4f}")
    return model_obj

# ── MISA ──────────────────────────────────────────────────────────────
print("Training MISA...")
misa   = compile_misa(build_misa())
tr_m   = wrap_misa(train_ds)
va_m   = wrap_misa(val_ds)
misa   = train_external(misa,
             os.path.join(CHECKPOINT_DIR, "MISA_best.keras"),
             tr_m, va_m, monitor="val_main_loss")
keras.backend.clear_session()

# ── CTNet ─────────────────────────────────────────────────────────────
print("Training CTNet...")
ctnet  = compile_ctnet(build_ctnet())
ctnet  = train_external(ctnet,
             os.path.join(CHECKPOINT_DIR, "CTNet_best.keras"),
             train_ds, val_ds, monitor="val_loss")   # CTNet uses val_loss
keras.backend.clear_session()

# ── UniMSE ────────────────────────────────────────────────────────────
print("Training UniMSE...")
unimse = compile_unimse(build_unimse())
tr_u   = wrap_unimse(train_ds)
va_u   = wrap_unimse(val_ds)
unimse = train_external(unimse,
             os.path.join(CHECKPOINT_DIR, "UniMSE_best.keras"),
             tr_u, va_u, monitor="val_main_loss")
keras.backend.clear_session()

print("\n All 3 external SOTA models trained.")

Training MISA...
Epoch 1/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 37s 33ms/step - aux_a_loss: 0.7484 - aux_t_loss: 0.7906 - aux_v_loss: 0.7203 - loss: 2.1425 - main_loss: 0.7492 - val_aux_a_loss: 0.1281 - val_aux_t_loss: 0.1275 - val_aux_v_loss: 0.1291 - val_loss: 0.4324 - val_main_loss: 0.1235 - learning_rate: 0.0010
Epoch 2/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - aux_a_loss: 0.1424 - aux_t_loss: 0.1415 - aux_v_loss: 0.1421 - loss: 0.4338 - main_loss: 0.1356 - val_aux_a_loss: 0.1243 - val_aux_t_loss: 0.1242 - val_aux_v_loss: 0.1247 - val_loss: 0.3640 - val_main_loss: 0.1139 - learning_rate: 0.0010
Epoch 3/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - aux_a_loss: 0.1333 - aux_t_loss: 0.1324 - aux_v_loss: 0.1294 - loss: 0.3722 - main_loss: 0.1224 - val_aux_a_loss: 0.1233 - val_aux_t_loss: 0.1228 - val_aux_v_loss: 0.1150 - val_loss: 0.3313 - val_main_loss: 0.1054 - learning_rate: 0.0010
Epoch 4/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - aux_a_loss: 0.1304 - aux_t_loss: 0.1279 - aux_v_los

/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis 3 of a tensor of shape (None, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


509/509 ━━━━━━━━━━━━━━━━━━━━ 64s 54ms/step - loss: 0.8008 - mae: 0.6192 - val_loss: 0.1797 - val_mae: 0.2194 - learning_rate: 0.0010
Epoch 2/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.1926 - mae: 0.2231 - val_loss: 0.1648 - val_mae: 0.1880 - learning_rate: 0.0010
Epoch 3/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 0.1709 - mae: 0.1956 - val_loss: 0.1506 - val_mae: 0.1730 - learning_rate: 0.0010
Epoch 4/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.1575 - mae: 0.1855 - val_loss: 0.1408 - val_mae: 0.1700 - learning_rate: 0.0010
Epoch 5/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - loss: 0.1461 - mae: 0.1817 - val_loss: 0.1352 - val_mae: 0.1708 - learning_rate: 0.0010
Epoch 6/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.1377 - mae: 0.1806 - val_loss: 0.1268 - val_mae: 0.1684 - learning_rate: 0.0010
Epoch 7/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - loss: 0.1297 - mae: 0.1786 - val_loss: 0.1220 - val_mae: 0.1652 - learning_rate: 0.0010
Epoch 8/5

In [ ]:
# ══ CELL S4 — Full 9-Model Comparison Table ══
import numpy as np, pandas as pd, os, keras
from sklearn.metrics import mean_squared_error, mean_absolute_error, f1_score
from sota_models import SOTA_CUSTOM_OBJECTS
from config import CHECKPOINT_DIR, RESULTS_DIR, EMOTION_NAMES, EMOTION_PRESENCE_THRESHOLD, N_EMOTIONS

def eval_external(model, name):
    all_true, all_pred = [], []
    for inputs, labels in test_ds:
        out = model(inputs, training=False)
        pred = out['main'] if isinstance(out,dict) else out
        all_pred.append(pred.numpy()); all_true.append(labels.numpy())
    y_true=np.vstack(all_true); y_pred=np.vstack(all_pred)
    rmse=float(np.sqrt(mean_squared_error(y_true,y_pred)))
    mae =float(mean_absolute_error(y_true,y_pred))
    yb=(y_true>=EMOTION_PRESENCE_THRESHOLD).astype(int).flatten()
    pb=(y_pred>=EMOTION_PRESENCE_THRESHOLD).astype(int).flatten()
    f1=float(f1_score(yb,pb,average='weighted',zero_division=0))
    print(f'  {name:<10} RMSE={rmse:.4f} MAE={mae:.4f} F1={f1:.4f}')
    return{'Model':name,'Fusion':'paper-arch','MSE':round(rmse**2,4),'MAE':round(mae,4),'RMSE':round(rmse,4),'F1':round(f1,4)}

misa  =keras.models.load_model(os.path.join(CHECKPOINT_DIR,'MISA_best.keras'),  custom_objects=SOTA_CUSTOM_OBJECTS)
ctnet =keras.models.load_model(os.path.join(CHECKPOINT_DIR,'CTNet_best.keras'))
unimse=keras.models.load_model(os.path.join(CHECKPOINT_DIR,'UniMSE_best.keras'))

print('External SOTA results:')
misa_row   = eval_external(misa,  'MISA')
ctnet_row  = eval_external(ctnet, 'CTNet')
unimse_row = eval_external(unimse,'UniMSE')

framework_df = pd.read_csv(os.path.join(RESULTS_DIR,'full_comparison.csv'))
full_df = pd.concat([framework_df, pd.DataFrame([misa_row,ctnet_row,unimse_row,chfn_row])],ignore_index=True)
full_df = full_df.sort_values('RMSE').reset_index(drop=True); full_df.index+=1
full_df.to_csv(os.path.join(RESULTS_DIR,'full_9model_comparison.csv'),index=False)
best_rmse=full_df['RMSE'].min(); best_f1=full_df['F1'].max()
print('\n'+'='*75)
print(f'{"#":<4} {"Model":<16} {"Fusion":<16} {"RMSE":>8} {"MAE":>8} {"F1":>8}  Tag')
print('-'*75)
for idx,row in full_df.iterrows():
    tag=' ◀ PROPOSED' if 'CHFN' in str(row['Model']) else ''
    tag+=' [EXT]' if row['Model'] in('MISA','CTNet','UniMSE') else ''
    tag+=' ⭐' if row['RMSE']==best_rmse else ''
    print(f'{idx:<4} {row["Model"]:<16} {row["Fusion"]:<16} {row["RMSE"]:>8.4f} {row["MAE"]:>8.4f} {row["F1"]:>8.4f} {tag}')
print('='*75)

External SOTA results:
  MISA       RMSE=0.2995 MAE=0.1660 F1=0.9266


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis 3 of a tensor of shape (32, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis 3 of a tensor of shape (11, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


  CTNet      RMSE=0.3043 MAE=0.1661 F1=0.9301


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis 3 of a tensor of shape (32, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


  UniMSE     RMSE=0.3013 MAE=0.1669 F1=0.9284

#    Model            Fusion               RMSE      MAE       F1  Tag
---------------------------------------------------------------------------
1    SelfMM           late               0.2776   0.1627   0.9283  ⭐
2    SelfMM           cross_modal        0.2786   0.1618   0.9287 
3    MulT             early              0.2787   0.1642   0.9303 
4    MFN              late               0.2788   0.1657   0.9287 
5    LMF              late               0.2792   0.1673   0.9287 
6    MFN              cross_modal        0.2793   0.1652   0.9293 
7    SelfMM           early              0.2794   0.1627   0.9279 
8    TFN              late               0.2795   0.1646   0.9289 
9    MFN              early              0.2797   0.1630   0.9286 
10   MulT             late               0.2802   0.1667   0.9292 
11   MulT             cross_modal        0.2816   0.1668   0.9288 
12   CHFN (Ours)      hierarchical       0.2821   0.1621   0.9287  

/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis 3 of a tensor of shape (11, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


In [ ]:
# ══ Register missing classes from novel_model into current session ══
import keras
from keras import layers, ops
import tensorflow as tf

# These are already in your uploaded novel_model.py but the old file
# on Drive doesn't have them yet — define them here for this session

@keras.saving.register_keras_serializable()
class ScaleOutput(layers.Layer):
    def __init__(self, scale=3.0, **kw):
        super().__init__(**kw); self.scale = scale
    def call(self, x): return x * self.scale
    def get_config(self):
        base = super().get_config(); base.update({"scale": self.scale}); return base

@keras.saving.register_keras_serializable()
class StackTensors(layers.Layer):
    def __init__(self, axis=1, **kw):
        super().__init__(**kw); self.axis = axis
    def call(self, inputs): return tf.stack(inputs, axis=self.axis)
    def compute_output_shape(self, input_shape):
        return (input_shape[0][0], len(input_shape), input_shape[0][-1])
    def get_config(self):
        base = super().get_config(); base.update({"axis": self.axis}); return base

@keras.saving.register_keras_serializable()
class ReduceMeanLayer(layers.Layer):
    def __init__(self, axis=1, **kw):
        super().__init__(**kw); self.axis = axis
    def call(self, x): return tf.reduce_mean(x, axis=self.axis)
    def compute_output_shape(self, input_shape):
        shape = list(input_shape); shape.pop(self.axis); return tuple(shape)
    def get_config(self):
        base = super().get_config(); base.update({"axis": self.axis}); return base

@keras.saving.register_keras_serializable()
class ReduceStdLayer(layers.Layer):
    def __init__(self, axis=1, **kw):
        super().__init__(**kw); self.axis = axis
    def call(self, x): return tf.math.reduce_std(x, axis=self.axis)
    def compute_output_shape(self, input_shape):
        shape = list(input_shape); shape.pop(self.axis); return tuple(shape)
    def get_config(self):
        base = super().get_config(); base.update({"axis": self.axis}); return base

@keras.saving.register_keras_serializable()
class ContrastiveAlignmentLoss(layers.Layer):
    def __init__(self, temperature=0.07, **kw):
        super().__init__(**kw); self.temperature = temperature
    def call(self, inputs, training=None):
        shar_t, shar_a, shar_v = inputs
        nt = tf.nn.l2_normalize(shar_t, axis=-1)
        na = tf.nn.l2_normalize(shar_a, axis=-1)
        nv = tf.nn.l2_normalize(shar_v, axis=-1)
        sim_ta = tf.reduce_mean(tf.reduce_sum(nt * na, axis=-1))
        sim_tv = tf.reduce_mean(tf.reduce_sum(nt * nv, axis=-1))
        sim_av = tf.reduce_mean(tf.reduce_sum(na * nv, axis=-1))
        self.add_loss((3.0 - sim_ta - sim_tv - sim_av) / 3.0 / self.temperature * 0.1)
        return inputs[0]
    def get_config(self):
        base = super().get_config(); base.update({"temperature": self.temperature}); return base

@keras.saving.register_keras_serializable()
class PairwiseLMF(layers.Layer):
    def __init__(self, rank=4, out_dim=64, **kw):
        super().__init__(**kw); self.rank = rank; self.out_dim = out_dim
    def build(self, input_shape):
        d = input_shape[0][-1]; init = keras.initializers.GlorotUniform()
        self.f1 = self.add_weight(shape=(self.rank, d+1, self.out_dim), name="f1", initializer=init)
        self.f2 = self.add_weight(shape=(self.rank, d+1, self.out_dim), name="f2", initializer=init)
        self.fw = self.add_weight(shape=(self.rank,), name="fw", initializer="ones")
        super().build(input_shape)
    def call(self, inputs):
        h1, h2 = inputs
        ones = ops.ones(ops.stack([ops.shape(h1)[0], 1]), dtype=h1.dtype)
        h1 = ops.concatenate([h1, ones], axis=-1)
        h2 = ops.concatenate([h2, ones], axis=-1)
        f1 = ops.einsum("bi,rio->bro", h1, self.f1)
        f2 = ops.einsum("bi,rio->bro", h2, self.f2)
        w  = ops.softmax(self.fw)
        return ops.relu(ops.einsum("bro,r->bo", f1 * f2, w))
    def get_config(self):
        base = super().get_config(); base.update({"rank": self.rank, "out_dim": self.out_dim}); return base

print(" All custom classes registered now re-run the CHFN v2 cell")

 All custom classes registered now re-run the CHFN v2 cell


In [ ]:
# ══ CHFN v2 — fully self-contained, no novel_model imports ══
import os, time, keras, tensorflow as tf
from keras import layers, ops, regularizers
import numpy as np
from models import AppendOne, LMFFusionLayer, CUSTOM_OBJECTS
from config import (TEXT_DIM, AUDIO_DIM, VISUAL_DIM, COMMON_DIM,
                    DROPOUT, WEIGHT_DECAY, N_EMOTIONS,
                    CHECKPOINT_DIR, RESULTS_DIR, EPOCHS,
                    PATIENCE, LR_PATIENCE, LR_FACTOR)

L2 = regularizers.l2(WEIGHT_DECAY)
N_BAG = 3
CHFN_CKPT_V2 = os.path.join(CHECKPOINT_DIR, "CHFN_v2_best.keras")

# ── Custom layers (all registered, no Lambda) ─────────────────────────────────

@keras.saving.register_keras_serializable()
class ScaleOutput(layers.Layer):
    def __init__(self, scale=3.0, **kw):
        super().__init__(**kw); self.scale = scale
    def call(self, x): return x * self.scale
    def get_config(self):
        base = super().get_config(); base.update({"scale": self.scale}); return base

@keras.saving.register_keras_serializable()
class StackTensors(layers.Layer):
    def __init__(self, axis=1, **kw):
        super().__init__(**kw); self.axis = axis
    def call(self, inputs): return tf.stack(inputs, axis=self.axis)
    def compute_output_shape(self, input_shape):
        return (input_shape[0][0], len(input_shape), input_shape[0][-1])
    def get_config(self):
        base = super().get_config(); base.update({"axis": self.axis}); return base

@keras.saving.register_keras_serializable()
class ReduceMeanLayer(layers.Layer):
    def __init__(self, axis=1, **kw):
        super().__init__(**kw); self.axis = axis
    def call(self, x): return tf.reduce_mean(x, axis=self.axis)
    def compute_output_shape(self, input_shape):
        shape = list(input_shape); shape.pop(self.axis); return tuple(shape)
    def get_config(self):
        base = super().get_config(); base.update({"axis": self.axis}); return base

@keras.saving.register_keras_serializable()
class ReduceStdLayer(layers.Layer):
    def __init__(self, axis=1, **kw):
        super().__init__(**kw); self.axis = axis
    def call(self, x): return tf.math.reduce_std(x, axis=self.axis)
    def compute_output_shape(self, input_shape):
        shape = list(input_shape); shape.pop(self.axis); return tuple(shape)
    def get_config(self):
        base = super().get_config(); base.update({"axis": self.axis}); return base

@keras.saving.register_keras_serializable()
class ContrastiveAlignmentLoss(layers.Layer):
    def __init__(self, temperature=0.07, **kw):
        super().__init__(**kw); self.temperature = temperature
    def call(self, inputs, training=None):
        st, sa, sv = inputs
        nt = tf.nn.l2_normalize(st, axis=-1)
        na = tf.nn.l2_normalize(sa, axis=-1)
        nv = tf.nn.l2_normalize(sv, axis=-1)
        sim_ta = tf.reduce_mean(tf.reduce_sum(nt*na, axis=-1))
        sim_tv = tf.reduce_mean(tf.reduce_sum(nt*nv, axis=-1))
        sim_av = tf.reduce_mean(tf.reduce_sum(na*nv, axis=-1))
        self.add_loss((3.0 - sim_ta - sim_tv - sim_av) / 3.0 / self.temperature * 0.1)
        return inputs[0]
    def get_config(self):
        base = super().get_config(); base.update({"temperature": self.temperature}); return base

@keras.saving.register_keras_serializable()
class PairwiseLMF(layers.Layer):
    def __init__(self, rank=4, out_dim=64, **kw):
        super().__init__(**kw); self.rank = rank; self.out_dim = out_dim
    def build(self, input_shape):
        d = input_shape[0][-1]; init = keras.initializers.GlorotUniform()
        self.f1 = self.add_weight(shape=(self.rank, d+1, self.out_dim), name="f1", initializer=init)
        self.f2 = self.add_weight(shape=(self.rank, d+1, self.out_dim), name="f2", initializer=init)
        self.fw = self.add_weight(shape=(self.rank,), name="fw", initializer="ones")
        super().build(input_shape)
    def call(self, inputs):
        h1, h2 = inputs
        ones = ops.ones(ops.stack([ops.shape(h1)[0], 1]), dtype=h1.dtype)
        h1 = ops.concatenate([h1, ones], axis=-1)
        h2 = ops.concatenate([h2, ones], axis=-1)
        f1 = ops.einsum("bi,rio->bro", h1, self.f1)
        f2 = ops.einsum("bi,rio->bro", h2, self.f2)
        w  = ops.softmax(self.fw)
        return ops.relu(ops.einsum("bro,r->bo", f1 * f2, w))
    def get_config(self):
        base = super().get_config(); base.update({"rank": self.rank, "out_dim": self.out_dim}); return base

CHFN_V2_CUSTOM_OBJECTS = {
    **CUSTOM_OBJECTS,
    "ScaleOutput":              ScaleOutput,
    "StackTensors":             StackTensors,
    "ReduceMeanLayer":          ReduceMeanLayer,
    "ReduceStdLayer":           ReduceStdLayer,
    "ContrastiveAlignmentLoss": ContrastiveAlignmentLoss,
    "PairwiseLMF":              PairwiseLMF,
}

# ── Helper functions ──────────────────────────────────────────────────────────

def _shared_private_encoder(x, tag):
    PRIV = SHAR = 64
    priv = layers.Dense(PRIV*2, activation="relu", kernel_regularizer=L2, name=f"{tag}_priv1")(x)
    priv = layers.BatchNormalization(name=f"{tag}_privbn")(priv)
    priv = layers.Dropout(DROPOUT, name=f"{tag}_privdrop")(priv)
    priv = layers.Dense(PRIV, activation="relu", name=f"{tag}_priv2")(priv)
    shar = layers.Dense(SHAR*2, activation="relu", kernel_regularizer=L2, name=f"{tag}_shar1")(x)
    shar = layers.BatchNormalization(name=f"{tag}_sharbn")(shar)
    shar = layers.Dropout(DROPOUT, name=f"{tag}_shardrop")(shar)
    shar = layers.Dense(SHAR, activation="relu", name=f"{tag}_shar2")(shar)
    enc  = layers.Concatenate(name=f"{tag}_enc")([priv, shar])
    return enc, priv, shar

def _hierarchical_fusion(et, ea, ev, tag):
    fta = PairwiseLMF(rank=4, out_dim=64, name=f"{tag}_lmf_ta")([et, ea])
    ftv = PairwiseLMF(rank=4, out_dim=64, name=f"{tag}_lmf_tv")([et, ev])
    fav = PairwiseLMF(rank=4, out_dim=64, name=f"{tag}_lmf_av")([ea, ev])
    ht  = AppendOne(name=f"{tag}_ht")(fta)
    ha  = AppendOne(name=f"{tag}_ha")(ftv)
    hv  = AppendOne(name=f"{tag}_hv")(fav)
    tri = LMFFusionLayer(rank=4, out_dim=64, name=f"{tag}_triple")([ht, ha, hv])
    return layers.Dense(COMMON_DIM, activation="relu", name=f"{tag}_expand")(tri)

def _bag_head(x, tag):
    x = layers.Dense(128, activation="relu", kernel_regularizer=L2, name=f"{tag}_b1")(x)
    x = layers.BatchNormalization(name=f"{tag}_b1bn")(x)
    x = layers.Dropout(DROPOUT, name=f"{tag}_b1drop")(x)
    x = layers.Dense(64,  activation="relu", name=f"{tag}_b2")(x)
    x = layers.Dropout(DROPOUT, name=f"{tag}_b2drop")(x)
    x = layers.Dense(32,  activation="relu", name=f"{tag}_b3")(x)
    x = layers.Dense(N_EMOTIONS, name=f"{tag}_logits")(x)
    x = layers.Activation("sigmoid", name=f"{tag}_sig")(x)
    return ScaleOutput(scale=3.0, name=f"{tag}_out")(x)

def _aux_head(x, name):
    x = layers.Dense(64, activation="relu", name=f"{name}_h1")(x)
    x = layers.Dropout(DROPOUT, name=f"{name}_drop")(x)
    x = layers.Dense(N_EMOTIONS, name=f"{name}_logits")(x)
    x = layers.Activation("sigmoid", name=f"{name}_sig")(x)
    return ScaleOutput(scale=3.0, name=name)(x)

# ── Build model ───────────────────────────────────────────────────────────────

def build_chfn_v2():
    tag   = "chfn2"
    inp_t = keras.Input(shape=(TEXT_DIM,),   name="text")
    inp_a = keras.Input(shape=(AUDIO_DIM,),  name="audio")
    inp_v = keras.Input(shape=(VISUAL_DIM,), name="visual")

    # Block 1: Shared-Private Encoding
    enc_t, _, shar_t = _shared_private_encoder(inp_t, f"{tag}_t")
    enc_a, _, shar_a = _shared_private_encoder(inp_a, f"{tag}_a")
    enc_v, _, shar_v = _shared_private_encoder(inp_v, f"{tag}_v")

    # Block 2: Contrastive alignment
    shar_t = ContrastiveAlignmentLoss(0.07, name=f"{tag}_contrast")(
        [shar_t, shar_a, shar_v])

    # Block 3: Hierarchical LMF fusion
    hier = _hierarchical_fusion(enc_t, enc_a, enc_v, f"{tag}_hier")

    # Block 4: 3-token self-attention (FIXED — softmax over 3 tokens not 1)
    seq  = StackTensors(axis=1, name=f"{tag}_seq")([enc_t, enc_a, enc_v])  # (B,3,128)
    attn = layers.MultiHeadAttention(
        num_heads=4, key_dim=COMMON_DIM//4,
        dropout=DROPOUT, name=f"{tag}_mha")(seq, seq)
    attn = layers.Add(name=f"{tag}_attn_add")([seq, attn])
    attn = layers.LayerNormalization(name=f"{tag}_attn_ln")(attn)
    ff   = layers.Dense(COMMON_DIM*2, activation="relu", name=f"{tag}_ff1")(attn)
    ff   = layers.Dense(COMMON_DIM,   name=f"{tag}_ff2")(ff)
    attn = layers.Add(name=f"{tag}_ff_add")([attn, ff])
    attn = layers.LayerNormalization(name=f"{tag}_ff_ln")(attn)
    attn_pooled = ReduceMeanLayer(axis=1, name=f"{tag}_pool")(attn)         # (B,128)

    # MFN-style gate
    gate_inp = layers.Concatenate(name=f"{tag}_ginp")([enc_t, enc_a, enc_v])
    gate     = layers.Dense(COMMON_DIM, activation="sigmoid", name=f"{tag}_gate")(gate_inp)
    attn_g   = layers.Multiply(name=f"{tag}_gated")([gate, attn_pooled])

    # Block 5: Combine + skip connection
    skip = ReduceMeanLayer(axis=1, name=f"{tag}_skip")(
           StackTensors(axis=1, name=f"{tag}_skip_stk")([enc_t, enc_a, enc_v]))
    combined = layers.Add(name=f"{tag}_combine")([hier, attn_g, skip])
    combined = layers.BatchNormalization(name=f"{tag}_combbn")(combined)
    combined = layers.Dropout(DROPOUT, name=f"{tag}_combdrop")(combined)
    combined = layers.Dense(COMMON_DIM, activation="relu", name=f"{tag}_combfc")(combined)

    # Block 6: Bagging heads
    heads   = [_bag_head(combined, f"{tag}_bag{i}") for i in range(N_BAG)]
    stacked = StackTensors(axis=1, name=f"{tag}_stack")(heads)
    main    = ReduceMeanLayer(axis=1, name="main")(stacked)
    unc     = ReduceStdLayer(axis=1,  name="uncertainty")(stacked)

    outputs = {"main": main, "uncertainty": unc,
               "aux_t": _aux_head(enc_t, "aux_t"),
               "aux_a": _aux_head(enc_a, "aux_a"),
               "aux_v": _aux_head(enc_v, "aux_v"),
               **{f"head_{i}": heads[i] for i in range(N_BAG)}}

    return keras.Model([inp_t, inp_a, inp_v], outputs, name="CHFN_v2")

# ── Dataset wrapper ───────────────────────────────────────────────────────────

def wrap_chfn(dataset):
    def _expand(inputs, labels):
        t = {"main": labels, "uncertainty": labels,
             "aux_t": labels, "aux_a": labels, "aux_v": labels}
        for i in range(N_BAG): t[f"head_{i}"] = labels
        return inputs, t
    return dataset.map(_expand)

# ── Train ─────────────────────────────────────────────────────────────────────

if os.path.exists(CHFN_CKPT_V2):
    print("Loading CHFN v2 checkpoint...")
    chfn = keras.models.load_model(CHFN_CKPT_V2, custom_objects=CHFN_V2_CUSTOM_OBJECTS)
else:
    chfn = build_chfn_v2()
    print(f"CHFN v2 parameters: {chfn.count_params():,}")

    AW = 0.15; HW = 0.20
    loss_d = {"main":"mse","uncertainty":None,
              "aux_t":"mse","aux_a":"mse","aux_v":"mse"}
    wt_d   = {"main":1.0,"uncertainty":0.0,
              "aux_t":AW,"aux_a":AW,"aux_v":AW}
    for i in range(N_BAG):
        loss_d[f"head_{i}"] = "mse"; wt_d[f"head_{i}"] = HW

    chfn.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3, weight_decay=1e-4),
        loss=loss_d, loss_weights=wt_d)

    cbs = [
        keras.callbacks.ModelCheckpoint(
            CHFN_CKPT_V2, monitor="val_main_loss", mode="min",
            save_best_only=True, verbose=1),
        keras.callbacks.EarlyStopping(
            monitor="val_main_loss", mode="min",
            patience=PATIENCE, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_main_loss", mode="min",
            factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=1e-6, verbose=1),
    ]

    t0 = time.time()
    history = chfn.fit(wrap_chfn(train_ds), validation_data=wrap_chfn(val_ds),
                       epochs=EPOCHS, callbacks=cbs, verbose=1)
    print(f"\nDone in {(time.time()-t0)/60:.1f} min | "
          f"best val: {min(history.history['val_main_loss']):.4f}")

CHFN v2 parameters: 895,444
Epoch 1/50
509/509 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - aux_a_loss: 0.4323 - aux_t_loss: 0.3198 - aux_v_loss: 0.3201 - head_0_loss: 0.5495 - head_1_loss: 0.5774 - head_2_loss: 0.5039 - loss: 1.0709 - main_loss: 0.4477
Epoch 1: val_main_loss improved from None to 0.10901, saving model to /content/drive/MyDrive/checkpoints/CHFN_v2_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/checkpoints/CHFN_v2_best.keras
509/509 ━━━━━━━━━━━━━━━━━━━━ 70s 64ms/step - aux_a_loss: 0.2177 - aux_t_loss: 0.1892 - aux_v_loss: 0.1821 - head_0_loss: 0.2466 - head_1_loss: 0.2573 - head_2_loss: 0.2328 - loss: 0.5765 - main_loss: 0.2071 - val_aux_a_loss: 0.1309 - val_aux_t_loss: 0.1295 - val_aux_v_loss: 0.1230 - val_head_0_loss: 0.1113 - val_head_1_loss: 0.1123 - val_head_2_loss: 0.1139 - val_loss: 0.3617 - val_main_loss: 0.1090 - learning_rate: 0.0010
Epoch 2/50
506/509 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - aux_a_loss: 0.1420 - aux_t_loss: 0.1377 - aux_v_loss: 0.1307

In [ ]:
# ══ CELL S5 — Paper LaTeX Table ══
import pandas as pd, os
from config import RESULTS_DIR
df=pd.read_csv(os.path.join(RESULTS_DIR,'full_9model_comparison.csv')).sort_values('RMSE').reset_index(drop=True)
df.index+=1
best_rmse=df['RMSE'].min(); best_mae=df['MAE'].min(); best_f1=df['F1'].max()
def fmt(v,best): s=f'{v:.4f}'; return f'\\textbf{{{s}}}' if v==best else s
print('\\begin{table}[t]\n\\centering')
print('\\caption{Comparison on 6-emotion multimodal stress detection.}')
print('\\label{tab:comparison}\n\\begin{tabular}{llccc}\n\\toprule')
print('Model & Type & RMSE$\\downarrow$ & MAE$\\downarrow$ & F1$\\uparrow$ \\\\\n\\midrule')
for _,row in df.iterrows():
    mtype=('\\textbf{Proposed}' if 'CHFN' in str(row['Model'])
           else 'Ext.~SOTA' if row['Model'] in('MISA','CTNet','UniMSE') else 'Baseline')
    name=f'\\textbf{{{row["Model"]}}}' if 'CHFN' in str(row['Model']) else row['Model']
    print(f'{name} & {mtype} & {fmt(row["RMSE"],best_rmse)} & {fmt(row["MAE"],best_mae)} & {fmt(row["F1"],best_f1)} \\\\')
print('\\bottomrule\n\\end{tabular}\n\\end{table}')

\begin{table}[t]
\centering
\caption{Comparison on 6-emotion multimodal stress detection.}
\label{tab:comparison}
\begin{tabular}{llccc}
\toprule
Model & Type & RMSE$\downarrow$ & MAE$\downarrow$ & F1$\uparrow$ \\
\midrule
SelfMM & Baseline & \textbf{0.2776} & 0.1627 & 0.9283 \\
SelfMM & Baseline & 0.2786 & \textbf{0.1618} & 0.9287 \\
MulT & Baseline & 0.2787 & 0.1642 & \textbf{0.9303} \\
MFN & Baseline & 0.2788 & 0.1657 & 0.9287 \\
LMF & Baseline & 0.2792 & 0.1673 & 0.9287 \\
MFN & Baseline & 0.2793 & 0.1652 & 0.9293 \\
SelfMM & Baseline & 0.2794 & 0.1627 & 0.9279 \\
TFN & Baseline & 0.2795 & 0.1646 & 0.9289 \\
MFN & Baseline & 0.2797 & 0.1630 & 0.9286 \\
MulT & Baseline & 0.2802 & 0.1667 & 0.9292 \\
MulT & Baseline & 0.2816 & 0.1668 & 0.9288 \\
\textbf{CHFN (Ours)} & \textbf{Proposed} & 0.2821 & 0.1621 & 0.9287 \\
TFN & Baseline & 0.2822 & 0.1636 & \textbf{0.9303} \\
TFN & Baseline & 0.2824 & 0.1639 & 0.9287 \\
LMF & Baseline & 0.2828 & 0.1695 & 0.9269 \\
LMF & Baseline & 0.2830 & 0.

In [ ]:
# ══ CELL 14 — Session Summary ══
import os, glob
from config import CHECKPOINT_DIR, RESULTS_DIR
ckpts=glob.glob(os.path.join(CHECKPOINT_DIR,'*.keras'))
pngs =glob.glob(os.path.join(RESULTS_DIR,'*.png'))
csvs =glob.glob(os.path.join(RESULTS_DIR,'*.csv'))
print(f'Checkpoints : {len(ckpts)}')
print(f'Charts      : {len(pngs)}')
print(f'CSV results : {len(csvs)}')
for p in sorted(pngs): print(f'  {os.path.basename(p)}')

Checkpoints : 19
Charts      : 0
CSV results : 20


In [7]:
# Run this cell in Colab
import shutil, os

# Zip the framework folder
shutil.make_archive(
    '/content/drive/MyDrive/stress_framework_download',
    'zip',
    '/content/drive/MyDrive',
    'stress_framework'
)
print("Done! Download stress_framework_download.zip from your Drive")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive'